### Test Shotgun

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

from transformers import AutoModelForCausalLM, AutoTokenizer

from fastchat.utils import str_to_torch_dtype
from fastchat.model import get_conversation_template

from evaluation.inference_shotgun import shotgun_forward

from model.shotgun.lru_cache import ShotgunCache, ShotgunCacheConfig

In [2]:
bench_name = 'spec_bench'
model_path = '/home/zhiyao/work/Spec-Bench-Models/vicuna-7b-v1.3'
max_new_tokens = 1024
dtype = 'float16'

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=str_to_torch_dtype(dtype),
    low_cpu_mem_usage=True,
    device_map="auto"
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [3]:
conv = get_conversation_template("vicuna")
conv.append_message(conv.roles[0], "Can you tell me a short joke?")
conv.append_message(conv.roles[1], None)
conv.stop_str = "</s>"
prompt = conv.get_prompt()
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
shotgun_cache = ShotgunCache([
    ShotgunCacheConfig(
        prefix_capacity=2**20,
        followup_capacity=8,
        prefix_len=2,
        followup_len=2,
    )
])

# Run twice so that the second run uses the cache.
output_ids, new_token, step, accept_length_list = shotgun_forward(
    inputs,
    model,
    tokenizer,
    max_new_tokens,
    shotgun_cache,
)
output_ids, new_token, step, accept_length_list = shotgun_forward(
    inputs,
    model,
    tokenizer,
    max_new_tokens,
    shotgun_cache,
)

In [4]:
output_ids = output_ids[0][len(inputs.input_ids[0]):]
output = tokenizer.decode(
    output_ids,
    spaces_between_special_tokens=False,
)

In [5]:
print(output)
print(accept_length_list)

Sure! Here's a short joke for you:

Why was the math book sad?

Because it had too many problems.</s>
[29, 3]


### Test Shotgun Helper Functions

In [6]:
import numpy as np
from model.shotgun.lru_cache import ShotgunCache, ShotgunCacheConfig
from model.shotgun.shotgun import (
    get_chained_draft_tokens,
    make_chained_input_ids,
    make_chained_4d_attention_mask,
    make_chained_position_ids,
    verify_chained_drafts
)

In [7]:
shotgun_cache = ShotgunCache([
    ShotgunCacheConfig(
        prefix_capacity=2**20,
        followup_capacity=8,
        prefix_len=2,
        followup_len=2,
    )
])

shotgun_cache.update_cache([0, 0, 11, 11])
shotgun_cache.update_cache([0, 0, 22, 22])
shotgun_cache.update_cache([0, 0, 33, 33])
shotgun_cache.update_cache([11, 11, 111, 111])
shotgun_cache.update_cache([11, 11, 222, 222])
shotgun_cache.update_cache([111, 111, 1111, 1111])

In [8]:
prefix_ids = [0, 0]
drafts, sum_drafts_len = get_chained_draft_tokens(prefix_ids, shotgun_cache, 11)
correct_drafts = \
((), [
    ((11, 11), [
        ((111, 111), []),
        ((222, 222), []),
    ]),
    ((22, 22), []),
    ((33, 33), []),
])
assert drafts == correct_drafts


In [9]:
prefix_ids = [0, 0]
drafts, sum_drafts_len = get_chained_draft_tokens(prefix_ids, shotgun_cache, 100)
correct_drafts = \
((), [
    ((11, 11), [
        ((111, 111), [
            ((1111, 1111), []),
        ]),
        ((222, 222), []),
    ]),
    ((22, 22), []),
    ((33, 33), []),
])
assert drafts == correct_drafts

In [10]:
uncached_prefix_ids = np.array([0, 0])
input_ids = make_chained_input_ids(
    uncached_prefix_ids=uncached_prefix_ids,
    drafts_root=drafts,
    sum_drafts_len=sum_drafts_len,
)
correct_input_ids = np.array([0, 0, 11, 11, 111, 111, 1111, 1111, 222, 222, 22, 22, 33, 33])
assert np.array_equal(input_ids, correct_input_ids)

In [11]:
position_ids = make_chained_position_ids(
    cached_prefix_len=2,
    uncached_prefix_len=3,
    drafts_root=drafts,
    sum_drafts_len=sum_drafts_len
)
correct_position_ids = [
    2, 3, 4, # uncached prefix
    5, 6,    # (11, 11)
    7, 8,    # (111, 111)
    9, 10,   # (1111, 1111)
    7, 8,    # (222, 222)
    5, 6,    # (22, 22)
    5, 6     # (33, 33)
]
assert list(position_ids) == correct_position_ids

In [12]:
mask = make_chained_4d_attention_mask(
    cached_prefix_len=2,
    uncached_prefix_len=3,
    drafts_root=drafts,
    sum_drafts_len=sum_drafts_len
)
correct_mask = [
    [1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,], # uncached prefix
    [1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,], # uncached prefix
    [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,], # uncached prefix
    [1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,], # (11, 11)
    [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,], # (11, 11)
    [1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,], # (111, 111)
    [1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.,], # (111, 111)
    [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.,], # (1111, 1111)
    [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.,], # (1111, 1111)
    [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,], # (222, 222)
    [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.,], # (222, 222)
    [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,], # (22, 22)
    [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0.,], # (22, 22)
    [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,], # (33, 33)
    [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1.,], # (33, 33)
]
assert np.array_equal(mask, correct_mask)

In [13]:
next_tok = 22
sampled_ids = np.array([42, 42, 42, 42, 42, 42, 42, 42, 22, 43, 42, 42])
accepted_ids = verify_chained_drafts(
    drafts_root=drafts,
    sampled_ids=sampled_ids,
    next_tok=next_tok,
)
correct_accepted_ids = [22, 22, 43]
assert accepted_ids == correct_accepted_ids

In [14]:
next_tok = 22
sampled_ids = np.array([42, 42, 42, 42, 42, 42, 42, 42, 43, 42, 42, 42])
accepted_ids = verify_chained_drafts(
    drafts_root=drafts,
    sampled_ids=sampled_ids,
    next_tok=next_tok,
)
correct_accepted_ids = [22, 43]
assert accepted_ids == correct_accepted_ids

In [15]:
next_tok = 43
sampled_ids = np.array([42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42, 42])
accepted_ids = verify_chained_drafts(
    drafts_root=drafts,
    sampled_ids=sampled_ids,
    next_tok=next_tok,
)
correct_accepted_ids = [43]
assert accepted_ids == correct_accepted_ids

In [16]:
next_tok = 11
sampled_ids = np.array([11, 111, 111, 1111, 1111, 43, 42, 42, 42, 42, 42, 42])
accepted_ids = verify_chained_drafts(
    drafts_root=drafts,
    sampled_ids=sampled_ids,
    next_tok=next_tok,
)
correct_accepted_ids = [11, 11, 111, 111, 1111, 1111, 43]
assert accepted_ids == correct_accepted_ids


In [17]:
next_tok = 11
sampled_ids = np.array([11, 111, 111, 1111, 43, 42, 42, 42, 42, 42, 42, 42])
accepted_ids = verify_chained_drafts(
    drafts_root=drafts,
    sampled_ids=sampled_ids,
    next_tok=next_tok,
)
correct_accepted_ids = [11, 11, 111, 111, 1111, 43]
assert accepted_ids == correct_accepted_ids

In [18]:
next_tok = 11
sampled_ids = np.array([11, 111, 111, 43, 42, 42, 42, 42, 42, 42, 42, 42])
accepted_ids = verify_chained_drafts(
    drafts_root=drafts,
    sampled_ids=sampled_ids,
    next_tok=next_tok,
)
correct_accepted_ids = [11, 11, 111, 111, 43]
assert accepted_ids == correct_accepted_ids

### Measure Optimal Total Draft Length

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from fastchat.utils import str_to_torch_dtype

In [ ]:
bench_name = 'spec_bench'
model_path = '/home/zhiyao/work/Spec-Bench-Models/vicuna-7b-v1.3'
max_new_tokens = 1024
dtype = 'float16'

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=str_to_torch_dtype(dtype),
    low_cpu_mem_usage=True,
    device_map="auto"
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [3]:
input_text = """Lily the little rabbit woke up before sunrise, long before the dew had dried on the grass. She rubbed her sleepy eyes, stretched her long ears, and hopped out of her cozy burrow. The world was quiet except for the soft chirping of crickets saying goodbye to the night and the gentle rustle of the breeze through the wildflowers. Lily tucked a small red scarf around her neck, packed a snack of crunchy carrot sticks into her satchel, and whispered to her family, “I'm off on an adventure!” With a final wave, she bounded across the soft earth into the glowing morning light.

The green meadow was already waking up. Yellow buttercups bowed in greeting, and a curious squirrel peeked from a fallen log. As Lily hopped along, she met her friend Rupert the field mouse, who was busy gathering sunflower seeds. “Good morning, Lily!” he squeaked, offering a handful of seeds. Lily smiled and tucked a few into her satchel for later. A moment later, a bright bluebird landed on her shoulder and tickled her ear with its song. “Follow the winding path to the oak,” it sang. “Magic awaits you today.” Filled with excitement, Lily bounced onward, greeting every creature she passed.

Soon she reached the edge of the forest, where the great oak tree stood tall and proud. Its wide branches stretched toward the sky, and its roots curled like giant sleeping serpents beneath the mossy ground. There, nestled between two roots, lay a glossy pebble that shimmered in shades of pink and green. Lily's heart thumped as she picked it up. The moment her paw touched the stone, it glowed warmly and hummed like a tiny bell. Before her eyes, a small wooden door appeared in the trunk. “Oh!” whispered Lily, “this must be the hidden entrance.”

Taking a deep breath, Lily opened the door and stepped into a glowing tunnel. Soft blue mushrooms lined the walls, casting a gentle light, and dozens of fireflies danced overhead. The tunnel floor was carpeted with velvety moss that tickled Lily's paws as she walked. Every few steps, she thought she heard distant laughter—like the song of wind chimes on a summer evening. In the hush of the tunnel, Lily felt a thrill of wonder. She held the pebble tight, its light guiding her forward, until at last the tunnel widened into a spacious chamber.

At the far end of the chamber, a secret garden bloomed under a shimmering glass dome. Butterflies with wings like stained glass fluttered among giant daisies, and vines heavy with glowing berries draped from marble columns. A polite frog wearing a tiny green vest hopped forward and bowed. “Welcome, Lily,” he croaked softly. “I am Sir Croakus, guardian of the Glimmering Grove. Please taste our nectar berries—they are sweeter than morning sunshine.” Lily plucked a berry, bit into its soft flesh, and sighed with delight. “It's the best I've ever eaten,” she said, her whiskers quivering with joy.

Sir Croakus led Lily to a rainbow-colored pond where friendly fish blew perfect bubble rings in the water. Nearby, a family of snails slid along painted stones, leaving trails of sparkling pearls behind. Lily spent the afternoon playing hide-and-seek among towering ferns, sketching the flowers in the soft soil with a stick, and listening to the frog tell tales of moonlit dances. She even raced a ladybug in a gentle game of tag, laughing as the garden creatures cheered her on.

Just as the sun reached its highest point,"""

In [4]:
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")["input_ids"]
position_ids = torch.arange(input_ids.shape[1]).unsqueeze(0).to("cuda")
attention_mask = torch.ones_like(input_ids).to("cuda")
prefix_len = input_ids.shape[1]

In [5]:
model_kwargs = {}
model_kwargs["past_key_values"] = None
model_kwargs["use_cache"] = True
model_kwargs["return_dict"] = True
model_kwargs["input_ids"] = input_ids
model_kwargs["position_ids"] = position_ids
model_kwargs["attention_mask"] = attention_mask


In [6]:
model_output = model(**model_kwargs)

In [7]:
past_key_values = model_output.past_key_values

In [ ]:
test_time = []

for test_len in range(261):
    input_ids = torch.full((1, test_len), 42, device="cuda")
    position_ids = torch.arange(prefix_len, prefix_len + test_len, device="cuda").unsqueeze(0)
    attention_mask = torch.ones((1, prefix_len + test_len), device="cuda")

    torch.cuda.synchronize()
    time_start = time.time()

    for test_round in range(50):
        model_kwargs["input_ids"] = input_ids
        model_kwargs["position_ids"] = position_ids
        model_kwargs["attention_mask"] = attention_mask
        model_kwargs["past_key_values"] = past_key_values
        model_output = model(**model_kwargs)

    torch.cuda.synchronize()
    time_end = time.time()

    time_cost = time_end - time_start
    print(f"test_len: {test_len}, Time cost: {time_cost} seconds")

    test_time.append(time_cost)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(261), test_time)

# Add red hollow circles at specific points
plt.plot(128, test_time[128], 'ro', markersize=8, mfc='none')
plt.plot(141, test_time[141], 'ro', markersize=8, mfc='none')
plt.plot(256, test_time[256], 'ro', markersize=8, mfc='none')

# Add text labels under the circles
plt.text(128, test_time[128]-0.1, "128", ha='center', fontsize=10)
plt.text(141, test_time[141]-0.1, "141", ha='center', fontsize=10)
plt.text(256, test_time[256]-0.1, "256", ha='center', fontsize=10)

plt.xlabel('Total Draft Length', fontsize=12)
plt.ylabel('Time (seconds)', fontsize=12)
plt.title('Inference Time vs Total Draft Length\nAttending to 935 Tokens in KV Cache')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.xlim(0, 260)
plt.ylim(0, 2.5)
plt.grid(True)
plt.show()

